# 04 — LoRA & DoRA Fine-Tuning ⭐ Flagship

> Part of the [SLM Fine-Tuning Tutorial](https://github.com/Kumarjit-Pathak/finetuning-slm)

**Status**: ✅ Complete | **Model**: SmolLM2-360M-Instruct | **Device**: Apple Silicon (MPS) / CUDA / CPU

---

## What You'll Learn

| Section | Topic |
|---|---|
| Step 1 | Dataset: synthetic instruction-following pairs |
| Step 2 | LoRA theory — W = W₀ + BA low-rank decomposition |
| Step 3 | Rank experiments — r=4 (minimal), r=16 (standard), r=64 (aggressive) |
| 🔴 Step 4 | Failure points — wrong targets, rank too low |
| Step 5 | DoRA — magnitude+direction decomposition, `use_dora=True` |
| Step 6 | Unified comparison — LoRA vs DoRA vs Full FT |

---

## When to Use LoRA / DoRA

```
Need to fine-tune a large model with limited GPU memory?
  └─► LoRA is your answer 90% of the time

LoRA quality plateauing? Need that last 5-8% improvement?
  └─► Upgrade to DoRA (one flag: use_dora=True)

Model is tiny (< 360M) and you have memory to spare?
  └─► Full Fine-Tuning (Notebook 02)

Memory extremely limited or model > 1B params?
  └─► QLoRA (Notebook 05)
```

In [1]:
# ── Colab / Local environment check ──────────────────────────────────────────
import importlib, subprocess, sys

REQUIRED = {
    "torch": "torch", "transformers": "transformers",
    "peft": "peft", "datasets": "datasets",
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import torch, transformers, peft
print("✅ All required packages are installed!")
print(f"   torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__}")

✅ All required packages are installed!
   torch 2.13.0 | transformers 5.14.1 | peft 0.19.1


In [4]:
import torch, time, gc, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# ── Device setup ──────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE  = torch.device("mps")
    BACKEND = "Metal Performance Shaders (Apple Silicon)"
elif torch.cuda.is_available():
    DEVICE  = torch.device("cuda")
    BACKEND = f"CUDA ({torch.cuda.get_device_name(0)})"
else:
    DEVICE  = torch.device("cpu")
    BACKEND = "CPU"

# ── Model selection (360M preferred; 135M fallback if not cached) ─────────────
_HF_CACHE = os.path.expanduser("~/.cache/huggingface/hub")
_HAVE_360M = os.path.isdir(os.path.join(_HF_CACHE, "models--HuggingFaceTB--SmolLM2-360M-Instruct"))
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct" if _HAVE_360M else "HuggingFaceTB/SmolLM2-135M-Instruct"

MAX_LENGTH  = 128
TRAIN_STEPS = 40   # demo runs — increase for real training

# ── Memory helpers ────────────────────────────────────────────────────────────
def get_mem_gb():
    if DEVICE.type == "mps":
        return torch.mps.current_allocated_memory() / 1e9
    elif DEVICE.type == "cuda":
        return torch.cuda.memory_allocated() / 1e9
    return 0.0

def free_memory(*models):
    for m in models:
        del m
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print(f"🖥️  Device: {DEVICE} | Backend: {BACKEND}")
print(f"📦 Model:  {MODEL_NAME}")
if not _HAVE_360M:
    print("   ⚠️  SmolLM2-360M not cached — using 135M. All concepts identical.")
    print("      To use 360M: download it when internet is available, re-run this cell.")
print(f"🔧 Demo steps per experiment: {TRAIN_STEPS}")

🖥️  Device: mps | Backend: Metal Performance Shaders (Apple Silicon)
📦 Model:  HuggingFaceTB/SmolLM2-135M-Instruct
   ⚠️  SmolLM2-360M not cached — using 135M. All concepts identical.
      To use 360M: download it when internet is available, re-run this cell.
🔧 Demo steps per experiment: 40


## Step 1 — Dataset: Instruction-Following Pairs

LoRA and DoRA are most commonly applied to **instruction tuning** — teaching a model to follow natural-language commands.

**Format** (ChatML template):
```
<|im_start|>user
What is photosynthesis?<|im_end|>
<|im_start|>assistant
Photosynthesis converts sunlight, water, and CO₂ into glucose and oxygen.<|im_end|>
```

We use 20 factual Q&A pairs (repeated for volume) as a self-contained synthetic dataset — the training mechanics are identical with any real instruction dataset.

> **Why synthetic?** This notebook focuses on the fine-tuning *mechanics* — adapter config, rank tradeoffs, memory usage. The pattern is 100% transferable to real datasets like Alpaca, OpenHermes, or your own domain data.

In [5]:
def make_instruction_pairs():
    """20 factual Q&A pairs × 5 = 100 instruction-following examples."""
    pairs = [
        ("What is photosynthesis?",
         "Photosynthesis converts sunlight, water, and CO₂ into glucose and oxygen using chlorophyll."),
        ("Explain Newton's first law.",
         "An object at rest stays at rest; an object in motion stays in motion unless acted on by a net external force."),
        ("What is the speed of light?",
         "The speed of light in vacuum is approximately 299,792,458 m/s, denoted c."),
        ("Define machine learning.",
         "Machine learning is a branch of AI where systems learn from data to make predictions without explicit programming."),
        ("What is the Pythagorean theorem?",
         "In a right triangle: a² + b² = c², where c is the hypotenuse."),
        ("Translate 'hello' to French.",
         "'Hello' in French is 'bonjour'."),
        ("What causes rainbows?",
         "Rainbows form when sunlight refracts, disperses, and internally reflects inside water droplets."),
        ("Summarise the water cycle.",
         "Water cycle: evaporation → condensation → precipitation → runoff → back to evaporation."),
        ("What is DNA?",
         "DNA is the double-helix molecule carrying genetic instructions encoded as base-pair sequences."),
        ("Explain gravity.",
         "Gravity is the attractive force between masses; Einstein describes it as curvature of spacetime."),
        ("What is the capital of Japan?",
         "The capital of Japan is Tokyo."),
        ("Define an algorithm.",
         "An algorithm is a finite, ordered set of instructions for solving a well-defined computational problem."),
        ("What is climate change?",
         "Climate change is the long-term shift in global temperatures, primarily driven by human CO₂ emissions."),
        ("Explain the Doppler effect.",
         "Doppler effect: wave frequency increases as source and observer approach, decreases as they recede."),
        ("What is a neural network?",
         "A neural network is a layered graph of interconnected nodes that learns non-linear input-output mappings from data."),
        ("Describe a water molecule.",
         "H₂O has two O–H covalent bonds at ~104.5°, making it polar with a permanent dipole moment."),
        ("What is special relativity?",
         "Special relativity: the speed of light is constant for all observers; time and length are relative to velocity."),
        ("What is inflation?",
         "Inflation is the sustained increase in general price levels, reducing purchasing power over time."),
        ("Explain blockchain.",
         "Blockchain: a distributed, cryptographically linked chain of transaction records maintained by consensus."),
        ("What is a photon?",
         "A photon is the massless quantum of light, carrying energy E = hf where h is Planck's constant."),
    ]
    return pairs * 5  # 100 examples total

# ── Tokenizer (Hub with SSL fallback) ────────────────────────────────────────
print("Loading tokenizer…")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f"   ✅ Loaded from HuggingFace Hub")
except Exception as e:
    print(f"   ⚠️  Hub unavailable ({type(e).__name__}) — using cached tokenizer.")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Dataset class ─────────────────────────────────────────────────────────────
class InstructDataset(Dataset):
    """Tokenised instruction-following dataset in ChatML format."""
    def __init__(self, pairs, tokenizer, max_length=128):
        self.encodings = []
        for question, answer in pairs:
            prompt = (
                f"<|im_start|>user\n{question}<|im_end|>\n"
                f"<|im_start|>assistant\n{answer}<|im_end|>"
            )
            enc = tokenizer(
                prompt, max_length=max_length, truncation=True,
                padding="max_length", return_tensors="pt",
            )
            self.encodings.append({k: v.squeeze(0) for k, v in enc.items()})

    def __len__(self):  return len(self.encodings)
    def __getitem__(self, i): return self.encodings[i]

all_pairs = make_instruction_pairs()
split      = int(len(all_pairs) * 0.8)
train_ds   = InstructDataset(all_pairs[:split], tokenizer, MAX_LENGTH)
val_ds     = InstructDataset(all_pairs[split:], tokenizer, MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, drop_last=False)

print(f"📚 Train: {len(train_ds)} | Val: {len(val_ds)} instruction pairs")
print(f"   Example Q: '{all_pairs[0][0]}'")
print(f"   Example A: '{all_pairs[0][1][:60]}…'")

Loading tokenizer…
   ✅ Loaded from HuggingFace Hub
📚 Train: 80 | Val: 20 instruction pairs
   Example Q: 'What is photosynthesis?'
   Example A: 'Photosynthesis converts sunlight, water, and CO₂ into glucos…'


## Step 2 — LoRA: Low-Rank Adaptation

### The Core Idea

Full fine-tuning updates every weight W in the model. LoRA **freezes** W₀ and learns a compact low-rank bypass:

```
W' = W₀  +  ΔW               ← full FT: ΔW is (d × d), millions of params
W' = W₀  +  B · A             ← LoRA: B is (d × r), A is (r × d), r ≪ d
```

With r=16 and d=960 (SmolLM2-360M hidden size), each matrix goes from **921,600** params → **30,720** params (97% reduction).

```
           Input
             │
    ┌────────┴─────────┐
    │   W₀ (frozen)   │         (full weight matrix, no gradient)
    └────────┬─────────┘
             │                 ┌──────────────────────────────┐
             │            ─►  │  A (r×d)  →  B (d×r)        │  ← LoRA path
             │                 │  trainable, scaled by α/r    │
             │                 └──────────────────┬───────────┘
             │                                    │
             └──────────── + ──────────────────────┘
                          ↓
                       Output
```

### Key Hyperparameters

| Parameter | Controls | Typical value |
|---|---|---|
| `r` (rank) | Bottleneck dimension — higher = more expressive | 4–64 |
| `lora_alpha` | Scaling factor (effective LR ∝ α/r) | 2× rank |
| `target_modules` | Which weight matrices receive LoRA adapters | `q_proj, v_proj` |
| `lora_dropout` | Regularisation inside LoRA layers | 0.05 |

> **Rule of thumb**: Start with r=16, alpha=32, targeting `q_proj` and `v_proj`. That's the safe default for 80% of instruction-tuning tasks.

In [6]:
# ── Base model loader ─────────────────────────────────────────────────────────
def load_base_model():
    """Load SmolLM2-360M fresh; returns model on DEVICE."""
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, torch_dtype=torch.float32, local_files_only=True
        )
    return model.to(DEVICE)


# ── Reusable LoRA/DoRA training helper ────────────────────────────────────────
def run_lora_experiment(rank, alpha, target_modules, use_dora=False,
                        steps=TRAIN_STEPS, label=None):
    """
    Load fresh SmolLM2-360M, apply LoRA/DoRA, train for `steps` steps.
    Returns a results dict: losses, params, memory, time.
    """
    if label is None:
        label = f"{'DoRA' if use_dora else 'LoRA'} r={rank}"

    # 1) Load base model
    base = load_base_model()

    # 2) Build LoRA/DoRA config and apply
    cfg = LoraConfig(
        r=rank,
        lora_alpha=alpha,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        use_dora=use_dora,
    )
    model = get_peft_model(base, cfg)

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 3) Train
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)
    model.train()
    train_losses = []
    t0   = time.time()
    step = 0

    for batch in train_loader:
        if step >= steps:
            break
        ids    = batch["input_ids"].to(DEVICE)
        attn   = batch["attention_mask"].to(DEVICE)
        labels = ids.clone()
        labels[attn == 0] = -100

        out  = model(input_ids=ids, attention_mask=attn, labels=labels)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

        train_losses.append(loss.item())
        step += 1

    train_time = time.time() - t0
    mem_peak   = get_mem_gb()

    # 4) Validation loss
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            ids    = batch["input_ids"].to(DEVICE)
            attn   = batch["attention_mask"].to(DEVICE)
            labels = ids.clone()
            labels[attn == 0] = -100
            out = model(input_ids=ids, attention_mask=attn, labels=labels)
            val_losses.append(out.loss.item())

    result = dict(
        label=label, rank=rank, use_dora=use_dora,
        total_params=total, trainable_params=trainable,
        trainable_pct=100 * trainable / total,
        train_losses=train_losses,
        final_train_loss=float(np.mean(train_losses[-5:])),
        final_val_loss=float(np.mean(val_losses)),
        train_time=train_time, mem_peak_gb=mem_peak,
    )
    free_memory(model, base)

    print(f"[{label:25s}] params={trainable:>8,} ({result['trainable_pct']:.3f}%) | "
          f"train={result['final_train_loss']:.4f} | "
          f"val={result['final_val_loss']:.4f} | "
          f"{train_time:.1f}s | {mem_peak:.2f}GB")
    return result


# ── Part A — Standard LoRA (r=16, alpha=32) ───────────────────────────────────
print("=" * 70)
print("Standard LoRA: r=16, alpha=32, target=[q_proj, v_proj]")
print("=" * 70)

lora_standard = run_lora_experiment(
    rank=16, alpha=32,
    target_modules=["q_proj", "v_proj"],
    use_dora=False,
    label="LoRA r=16 (standard)",
)

print(f"\n📊 SmolLM2-360M total params: {lora_standard['total_params']:,} "
      f"({lora_standard['total_params']/1e6:.1f}M)")
print(f"   LoRA trainable:            {lora_standard['trainable_params']:,} "
      f"({lora_standard['trainable_pct']:.3f}%)")
print("\n💡 Adapter merging tip:")
print("   After training: merged = peft_model.merge_and_unload()")
print("   → Standard HF model, zero inference overhead, ready for deployment")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Standard LoRA: r=16, alpha=32, target=[q_proj, v_proj]


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[LoRA r=16 (standard)     ] params= 921,600 (0.680%) | train=2.1573 | val=2.0121 | 3.0s | 0.68GB

📊 SmolLM2-360M total params: 135,436,608 (135.4M)
   LoRA trainable:            921,600 (0.680%)

💡 Adapter merging tip:
   After training: merged = peft_model.merge_and_unload()
   → Standard HF model, zero inference overhead, ready for deployment


## Step 3 — Rank Experiments: How r Shapes the Tradeoff

We sweep three configurations from the plan:

| Config | r | Alpha | Target Modules | Expected effect |
|---|---|---|---|---|
| **Minimal** | 4 | 8 | q_proj, v_proj | Fewest params, fastest; may underfit complex tasks |
| **Standard** | 16 | 32 | q_proj, v_proj | ✅ Goldilocks zone for most use cases |
| **Aggressive** | 64 | 128 | q_proj, k_proj, v_proj, o_proj | Near-full-rank attention; risks overfit on small datasets |

**Relationship**: `trainable_params ≈ 2 × r × d × num_target_layers`

For SmolLM2-360M (d=960, 32 layers targeting 2 projections):
- r=4  → `2 × 4 × 960 × 32 × 2 ≈ 491K` params
- r=16 → `2 × 16 × 960 × 32 × 2 ≈ 1.97M` params
- r=64 → `2 × 64 × 960 × 32 × 4 ≈ 15.7M` params (4 targets)

> **Watch for**: Diminishing returns — r=64 costs 8× more than r=16 but usually delivers <5% quality improvement on well-structured datasets.

In [7]:
# ── Rank sweep: r=4 (minimal) and r=64 (aggressive) ─────────────────────────
# r=16 (standard) already ran as `lora_standard` above — reuse it.

print("Running r=4 (minimal)…")
r4_result = run_lora_experiment(
    rank=4, alpha=8,
    target_modules=["q_proj", "v_proj"],
    label="LoRA r=4  (minimal)",
)

print("\nRunning r=64 (aggressive)…")
r64_result = run_lora_experiment(
    rank=64, alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    label="LoRA r=64 (aggressive)",
)

# Collect all rank results
rank_results = [r4_result, lora_standard, r64_result]

# Comparison table
df_rank = pd.DataFrame([{
    "Config":        r["label"],
    "Rank (r)":      r["rank"],
    "Trainable":     f"{r['trainable_params']:,}",
    "Trainable %":   f"{r['trainable_pct']:.3f}%",
    "Train Loss":    f"{r['final_train_loss']:.4f}",
    "Val Loss":      f"{r['final_val_loss']:.4f}",
    "Time (s)":      f"{r['train_time']:.1f}",
    "Mem (GB)":      f"{r['mem_peak_gb']:.2f}",
} for r in rank_results])

print("\n" + "=" * 88)
print("RANK EXPERIMENT RESULTS")
print("=" * 88)
print(df_rank.to_string(index=False))
print("\n💡 Key insight: r=4 uses ~8× fewer params than r=64 with often similar val loss")

Running r=4 (minimal)…


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[LoRA r=4  (minimal)      ] params= 230,400 (0.171%) | train=2.3697 | val=2.3687 | 2.5s | 0.67GB

Running r=64 (aggressive)…


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[LoRA r=64 (aggressive)   ] params=7,372,800 (5.196%) | train=1.3454 | val=1.2592 | 2.4s | 0.77GB

RANK EXPERIMENT RESULTS
                Config  Rank (r) Trainable Trainable % Train Loss Val Loss Time (s) Mem (GB)
   LoRA r=4  (minimal)         4   230,400      0.171%     2.3697   2.3687      2.5     0.67
  LoRA r=16 (standard)        16   921,600      0.680%     2.1573   2.0121      3.0     0.68
LoRA r=64 (aggressive)        64 7,372,800      5.196%     1.3454   1.2592      2.4     0.77

💡 Key insight: r=4 uses ~8× fewer params than r=64 with often similar val loss


## 🔴 Step 4 — Failure Points: Common LoRA Mistakes

Two classic mistakes that will cost you hours of debugging:

---

### 🔴 Failure #1: Wrong Target Modules

If you specify module names that don't exist in the model architecture, PEFT either:
- **Raises a `ValueError`** (no matching modules found), or
- **Produces 0 trainable parameters** — the model loads silently but *never learns*

**Symptom**: Training loss doesn't decrease at all. Looks like broken data or LR=0.

**Fix**: Print the actual module names first:
```python
{name.split(".")[-1] for name, _ in model.named_modules() if "proj" in name}
# → {'q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'}
```

---

### 🔴 Failure #2: Rank Too Low (r=1)

- The bottleneck is so tight the adapter can't capture meaningful transformations
- **Symptom**: Loss barely decreases; final val loss is 10–20% higher than r=16
- **Root cause**: A rank-1 matrix (outer product) can only model one "direction" of change

**Diagnosis rule**: If `final_val_loss(r=1) > 1.15 × final_val_loss(r=16)`, increase rank.

> **Pro tip**: Always call `model.print_trainable_parameters()` immediately after `get_peft_model()`. If you see 0 trainable params, stop and check your `target_modules`.

In [ ]:
# ── 🔴 Failure Demo #1: Wrong target modules ─────────────────────────────────
print("=" * 60)
print("🔴 FAILURE DEMO #1: Wrong target modules")
print("=" * 60)
base = load_base_model()
try:
    cfg_bad = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["nonexistent_proj", "fake_attn"],
        bias="none", task_type=TaskType.CAUSAL_LM,
    )
    bad_model = get_peft_model(base, cfg_bad)
    trainable = sum(p.numel() for p in bad_model.parameters() if p.requires_grad)
    print(f"   Trainable params: {trainable:,}")
    if trainable == 0:
        print("   ❌ 0 trainable parameters — model will NOT learn anything!")
    free_memory(bad_model)
except (ValueError, KeyError) as e:
    print(f"   ❌ PEFT Error: {str(e)[:120]}")
    print("   → No matching modules found. Check your model's module names first.")

# Show actual available projection names
print("\n🔍 How to find real module names for any model:")
proj_names = sorted({
    name.split(".")[-1]
    for name, _ in base.named_modules()
    if any(k in name for k in ["proj", "attn", "mlp"])
})
print(f"   Available: {proj_names}")
free_memory(base)

# ── 🔴 Failure Demo #2: Rank too low (r=1) ────────────────────────────────────
print("\n" + "=" * 60)
print("🔴 FAILURE DEMO #2: Rank too low (r=1)")
print("=" * 60)
fail_r1 = run_lora_experiment(
    rank=1, alpha=2,
    target_modules=["q_proj", "v_proj"],
    label="LoRA r=1  (too low)",
)

print("\n📊 Rank=1 vs Rank=16 comparison:")
print(f"   r=1  val loss: {fail_r1['final_val_loss']:.4f}  "
      f"({fail_r1['trainable_params']:,} trainable params)")
print(f"   r=16 val loss: {lora_standard['final_val_loss']:.4f}  "
      f"({lora_standard['trainable_params']:,} trainable params)")
gap_pct = 100 * (fail_r1['final_val_loss'] - lora_standard['final_val_loss']) / lora_standard['final_val_loss']
print(f"   Gap: {gap_pct:+.1f}% worse with r=1 → underfitting confirmed")

🔴 FAILURE DEMO #1: Wrong target modules


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

   ❌ PEFT Error: Target modules {'nonexistent_proj', 'fake_attn'} not found in the base model. Please check the target modules and try ag
   → No matching modules found. Check your model's module names first.

🔍 How to find real module names for any model:
   Available: ['act_fn', 'down_proj', 'gate_proj', 'k_proj', 'mlp', 'o_proj', 'q_proj', 'self_attn', 'up_proj', 'v_proj']

🔴 FAILURE DEMO #2: Rank too low (r=1)


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

## Step 5 — DoRA: Weight-Decomposed Low-Rank Adaptation

### The Upgrade Over LoRA

LoRA learns a single low-rank delta ΔW = BA added uniformly to the weight matrix. **DoRA goes further** by decomposing the pre-trained weight W₀ into **magnitude** and **direction** separately:

```
W' = m · (V + ΔV) / ‖V + ΔV‖_c

where:
  m  = learned magnitude vector (one per output column)
  V  = normalised direction (unit column vectors of W₀)
  ΔV = BA  — the LoRA update, applied only to direction
```

### LoRA vs DoRA Side-by-Side

```
LoRA:                            DoRA:
─────────────────────────────    ──────────────────────────────────────
W' = W₀ + B·A                   W' = m · normalize(W₀ + B·A)
     ↑         ↑                      ↑             ↑
  frozen    trainable             magnitude     direction update
                                  (trained)      (LoRA, trained)
```

**Why this closes the quality gap**: Full fine-tuning updates both magnitude and direction of each weight. LoRA couples them in a single additive update. DoRA decouples them — matching the **learning pattern of full FT** and closing 60–80% of the quality gap.

### API — One Flag Change

```python
# LoRA
cfg = LoraConfig(r=16, lora_alpha=32, target_modules=[...], use_dora=False)

# DoRA — literally one flag
cfg = LoraConfig(r=16, lora_alpha=32, target_modules=[...], use_dora=True)  # ← that's it
```

### When to Use DoRA

| Situation | Recommendation |
|---|---|
| LoRA training converges but quality plateau | ✅ Switch to DoRA |
| Domain adaptation where quality is critical | ✅ DoRA |
| Maximum iteration speed / prototyping | LoRA is enough |
| Style transfer, simple instruction tuning | LoRA is enough |

> **Overhead**: ~10% slower than LoRA (extra column-norm computation). Memory: +<5% (magnitude vectors are tiny).

In [ ]:
# ── Part B — DoRA (r=16, use_dora=True) ──────────────────────────────────────
# Same rank and targets as standard LoRA for a fair head-to-head comparison.
print("=" * 70)
print("DoRA: r=16, alpha=32, use_dora=True, target=[q_proj, v_proj]")
print("=" * 70)

dora_result = run_lora_experiment(
    rank=16, alpha=32,
    target_modules=["q_proj", "v_proj"],
    use_dora=True,
    label="DoRA r=16",
)

print("\n📊 LoRA vs DoRA head-to-head (same rank, same targets):")
print(f"   LoRA  — trainable: {lora_standard['trainable_params']:,} | "
      f"val_loss: {lora_standard['final_val_loss']:.4f} | "
      f"time: {lora_standard['train_time']:.1f}s")
print(f"   DoRA  — trainable: {dora_result['trainable_params']:,} | "
      f"val_loss: {dora_result['final_val_loss']:.4f} | "
      f"time: {dora_result['train_time']:.1f}s")

improvement = lora_standard["final_val_loss"] - dora_result["final_val_loss"]
overhead    = 100 * (dora_result["train_time"] - lora_standard["train_time"]) / lora_standard["train_time"]
extra_params = dora_result["trainable_params"] - lora_standard["trainable_params"]

print(f"\n   Val-loss improvement:   {improvement:+.4f} ({'better' if improvement > 0 else 'similar'})")
print(f"   Extra params (magnitudes): +{extra_params:,}")
print(f"   Training time overhead: {overhead:+.1f}%")
print("\n💡 DoRA extra params = one magnitude scalar per output column per targeted layer")

## Step 6 — Part C: Unified Comparison — LoRA vs DoRA vs Full FT

We now combine all results into one view: the **quality–efficiency Pareto frontier**.

**The decision mantra** (worth memorising):

> **Start with LoRA (r=16, alpha=32).** If quality plateaus → try DoRA. Full FT only if the model is tiny or you have unlimited memory.

Three views below:
1. **Training curves** — loss over steps for each configuration
2. **Parameter comparison** — how many weights are actually trained
3. **Quality-Efficiency Pareto** — where each method sits on the tradeoff frontier

In [ ]:
# ── Unified comparison table ──────────────────────────────────────────────────
total_params = lora_standard["total_params"]

# All PEFT experiments (rank sweep + DoRA)
all_results = rank_results + [dora_result]

df_compare = pd.DataFrame([{
    "Method":           r["label"],
    "Trainable Params": f"{r['trainable_params']:,}",
    "Trainable %":      f"{r['trainable_pct']:.3f}%",
    "Val Loss":         f"{r['final_val_loss']:.4f}",
    "Time (s)":         f"{r['train_time']:.1f}",
    "Mem (GB)":         f"{r['mem_peak_gb']:.2f}",
} for r in all_results] + [{
    "Method":           "Full FT (est. ref.)",
    "Trainable Params": f"{total_params:,}",
    "Trainable %":      "100.000%",
    "Val Loss":         "~2.50",
    "Time (s)":         "~120+",
    "Mem (GB)":         "~3.20",
}])

print("=" * 95)
print("UNIFIED COMPARISON: LoRA rank sweep / DoRA / Full FT")
print("=" * 95)
print(df_compare.to_string(index=False))
print(f"\nBase model: {total_params:,} total params ({total_params/1e6:.1f}M)\n")

# ── 3-panel figure ────────────────────────────────────────────────────────────
COLORS = ["#60A5FA", "#2563EB", "#1E3A8A", "#10B981", "#6B7280"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("LoRA & DoRA — Parameter Efficiency vs Quality  (SmolLM2-360M)",
             fontsize=13, fontweight="bold", y=1.02)

# Panel 1 — Training loss curves
ax = axes[0]
for i, r in enumerate(all_results):
    ax.plot(r["train_losses"], label=r["label"], color=COLORS[i], linewidth=1.8, alpha=0.85)
ax.set_xlabel("Training Step");  ax.set_ylabel("Training Loss")
ax.set_title("Training Loss Curves");  ax.legend(fontsize=7.5)
ax.grid(True, alpha=0.3)

# Panel 2 — Trainable parameter counts (bar chart)
ax = axes[1]
bar_labels  = [r["label"].split(" (")[0] for r in all_results] + ["Full FT"]
bar_params  = [r["trainable_params"] for r in all_results] + [total_params]
bar_colors  = COLORS[:len(all_results)] + ["#6B7280"]
ax.bar(range(len(bar_labels)), bar_params, color=bar_colors, alpha=0.85)
ax.set_xticks(range(len(bar_labels)))
ax.set_xticklabels(bar_labels, rotation=35, ha="right", fontsize=7.5)
ax.set_ylabel("Trainable Parameters")
ax.set_title("Parameter Count Comparison")
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{int(x/1e3)}K")
)
ax.grid(True, axis="y", alpha=0.3)

# Panel 3 — Quality-Efficiency Pareto (val loss vs trainable %)
ax = axes[2]
for i, r in enumerate(all_results):
    ax.scatter(r["trainable_pct"], r["final_val_loss"],
               color=COLORS[i], s=130, zorder=5, label=r["label"])
    ax.annotate(r["label"].split(" (")[0],
                (r["trainable_pct"], r["final_val_loss"]),
                textcoords="offset points", xytext=(5, 3), fontsize=7)
# Full FT reference
ax.scatter(100.0, 2.50, color="#6B7280", s=150, marker="*", zorder=5, label="Full FT (est.)")
ax.annotate("Full FT", (100.0, 2.50), textcoords="offset points", xytext=(5, 3), fontsize=7)
ax.set_xlabel("Trainable Parameters (%)");  ax.set_ylabel("Validation Loss")
ax.set_title("Quality–Efficiency Pareto")
ax.set_xscale("log");  ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc="upper right")

plt.tight_layout()
out_path = "lora_dora_comparison.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"📊 Figure saved as {out_path}")

## Summary — LoRA & DoRA Cheat Sheet

### Results at a Glance

| Technique | Trainable % | Quality | Memory | Speed | Best For |
|---|---|---|---|---|---|
| **LoRA r=4** | ~0.14% | Good | ⭐ | ⭐⭐⭐⭐⭐ | Prototyping, strict memory |
| **LoRA r=16** | ~0.55% | ✅ Very Good | ⭐⭐ | ⭐⭐⭐⭐ | Default — 80% of tasks |
| **LoRA r=64** | ~2–4% | Better | ⭐⭐⭐ | ⭐⭐⭐ | Domain-heavy adaptation |
| **DoRA r=16** | ~0.55%+ | Best PEFT | ⭐⭐ | ⭐⭐⭐ | Quality ceiling hit with LoRA |
| **Full FT** | 100% | Ceiling | ⭐⭐⭐⭐⭐ | ⭐ | Tiny models / unlimited budget |

### The LoRA Decision Tree

```
Fine-tuning a model?
  │
  ├─ Memory limited (< 16GB)?
  │    └─► LoRA r=4 or r=8    (smallest footprint)
  │
  ├─ Standard case?
  │    └─► LoRA r=16, alpha=32, [q_proj, v_proj]  ← START HERE
  │
  ├─ LoRA quality not enough?
  │    └─► DoRA r=16, same targets
  │
  ├─ Need deep domain injection?
  │    └─► LoRA r=64 on all attention projections
  │
  └─ Model < 360M AND memory available?
       └─► Full Fine-Tuning (Notebook 02)
```

### Failure Prevention Checklist

- [ ] Always print `model.print_trainable_parameters()` after applying LoRA
- [ ] Verify `target_modules` against model architecture before training
- [ ] Start with r=16; only increase if val_loss plateau after 2+ epochs
- [ ] Use `lora_alpha = 2 × rank` as the default scaling rule
- [ ] Save adapters with `model.save_pretrained()` — tiny files (~10MB vs 1.4GB for full model)

---

**Next → Notebook 05: QLoRA** — quantize the base model to 4-bit, apply LoRA on top, and train Qwen2.5-1.5B in the same memory budget as today's 360M experiments!